# 第12回 演習: 量子通信と量子中継器 (QI#1)

この演習では、長距離量子通信を支える二つの道具 **もつれ交換 (entanglement swapping)** と
**純粋化 (entanglement purification)** を、密度行列レベルで実際に動かします。

扱う流れ:

- **演習1** ファイバ損失が距離に指数で効くことを体感する (中継器が要る理由)
- **演習2** 忠実度 `F = <Φ⁺|ρ|Φ⁺>` を密度行列から計算する
- **演習3** もつれ交換の忠実度 `F_swap` を数値的に確認する
- **演習4** 交換を鎖状に連ねると `F` が 1/4 へ落ちることを見る
- **演習5** 「同じ F でも状態は別物」を構成して確かめる
- **演習6** 純粋化の写像 (閾値 1/2・天井) を動かす
- **演習7** 交換と純粋化を入れ子にする (ノコギリ波と代金)

数式の完全な導出は副読本に送り、ここでは **動かして体感する** ことを優先します。

> 環境: `numpy` のみで完結します (qiskit は 演習3 の補足セルで任意使用)。


## 演習1: なぜ中継器が要るのか — ファイバ損失を体感する (→ スライド「距離の壁：指数減衰」)

光ファイバに光子を通すと、生き残る確率は距離 `L` に対して **指数で** 減ります。

$$P(L) = e^{-L / L_{\rm att}}, \qquad L_{\rm att} \approx 20\,\text{km}$$

1個のもつれペアを遠くへ届けるのに平均何回試行が要るか (= `1/P`) を計算すると、
**増幅もコピーもできない** 量子では単純な直送が破綻することが数値で分かります。
これが以降の中継器・交換・純粋化すべての動機です。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
jp = fm.FontProperties(fname="/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc")

L_att = 20.0  # 減衰長 [km] (光ファイバの目安)
def P_survive(L): return np.exp(-np.asarray(L, dtype=float) / L_att)

print(f"{'L[km]':>7} {'生存確率P':>14} {'平均試行 1/P':>16}")
for L in [20, 100, 500, 1000]:
    p = float(P_survive(L))
    print(f"{L:>7d} {p:>14.3e} {1/p:>16.1e}")

Ls = np.linspace(0, 1000, 300)
plt.figure(figsize=(7,4))
plt.semilogy(Ls, P_survive(Ls), color="#C00000", lw=2)
plt.axhline(1e-2, color="gray", ls="--", alpha=0.6)
plt.text(35, 1.4e-2, "1% 目安", color="gray", fontproperties=jp, fontsize=9)
plt.xlabel("距離 L [km]", fontproperties=jp)
plt.ylabel("1光子の生存確率 P (片対数)", fontproperties=jp)
plt.title("ファイバ損失は距離に指数で効く -> 単純な直送は破綻", fontproperties=jp)
plt.grid(alpha=0.3, which="both"); plt.tight_layout(); plt.show()

print("\n1000 km では 1 ペア届けるのに ~10^22 回。距離を分割して中継するしかない。")

## 演習2: 忠実度 F の計算 (→ スライド「純粋状態と混合状態」「忠実度 F」)

手元のペアの状態を密度行列 `ρ` で表し、理想ベルペア `|Φ⁺⟩` への重なり

$$F = \langle \Phi^+ | \rho | \Phi^+ \rangle$$

を計算します。まずベル基底と、雑音ペアのモデルとして **Werner 状態**

$$\rho_W(F) = F\,|\Phi^+\rangle\langle\Phi^+| + \frac{1-F}{3}\left(|\Phi^-\rangle\langle\Phi^-| + |\Psi^+\rangle\langle\Psi^+| + |\Psi^-\rangle\langle\Psi^-|\right)$$

を用意します。

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

# ベル状態 (列ベクトル)
s = 1/np.sqrt(2)
PhiP = np.array([1,0,0,1])*s   # |Φ⁺> = (|00>+|11>)/√2
PhiM = np.array([1,0,0,-1])*s  # |Φ⁻>
PsiP = np.array([0,1,1,0])*s   # |Ψ⁺>
PsiM = np.array([0,1,-1,0])*s  # |Ψ⁻>

def proj(v):
    v = v.reshape(-1,1)
    return v @ v.conj().T

P_PhiP, P_PhiM = proj(PhiP), proj(PhiM)
P_PsiP, P_PsiM = proj(PsiP), proj(PsiM)

def werner(F):
    "忠実度 F の Werner 状態 (雑音ペアの素直なモデル)"
    return F*P_PhiP + (1-F)/3*(P_PhiM + P_PsiP + P_PsiM)

def fidelity(rho):
    "F = <Φ⁺|ρ|Φ⁺>"
    return float(np.real(PhiP.conj() @ rho @ PhiP))

# 動かしてみる
for F in [1.0, 0.9, 0.5, 0.25]:
    rho = werner(F)
    print(f"入力F={F:.2f}  ->  計算F={fidelity(rho):.4f}  (trace={np.real(np.trace(rho)):.3f})")


**確認**: `F=1` は理想ベルペア (純粋状態)、`F=1/4` は完全混合 `I/4` (情報ゼロ) です。

> **やってみる**: `werner(0.7)` の密度行列 `print` してみましょう。対角に重みが乗り、
> 非対角に `|Φ⁺⟩` のコヒーレンスが残っているのが見えます。

## 演習3: もつれ交換の忠実度 (→ スライド「swap 機構」「交換後の忠実度」)

中継器 R1 が手元の2つの片割れに **ベル測定** をかけると、直接会っていない両端がもつれます。
脱分極(Werner)前提では、交換後の忠実度は

$$F_{\rm swap} = F_a F_b + \frac{(1-F_a)(1-F_b)}{3}$$

になります。これを **数値的なベル測定** で確かめます (公式を信じず、自分で再現する)。

In [ ]:
# 公式
def swap_formula(Fa, Fb):
    return Fa*Fb + (1-Fa)*(1-Fb)/3

# 数値: 4 qubit (A, r1, r2, B)。ペアa=(A,r1), ペアb=(r2,B)。
# r1,r2 にベル測定し、|Φ⁺> が出た分岐で A-B の F を読む。
bell = {'PhiP':PhiP, 'PhiM':PhiM, 'PsiP':PsiP, 'PsiM':PsiM}

def swap_numeric_phip(Fa, Fb):
    rho = np.kron(werner(Fa), werner(Fb))   # (A,r1,r2,B), 16x16
    I2 = np.eye(2)
    # r1,r2 に |Φ⁺> を射影
    M = np.kron(np.kron(I2, proj(PhiP)), I2)
    post = M @ rho @ M.conj().T
    p = float(np.real(np.trace(post)))
    post /= p
    # r1,r2 を partial trace して A-B を取り出す
    t = post.reshape(2,2,2,2, 2,2,2,2)
    rAB = np.zeros((4,4), dtype=complex)
    for A in range(2):
        for B in range(2):
            for Ap in range(2):
                for Bp in range(2):
                    rAB[2*A+B, 2*Ap+Bp] = sum(
                        t[A,r1,r2,B, Ap,r1,r2,Bp]
                        for r1 in range(2) for r2 in range(2))
    return float(np.real(PhiP.conj() @ rAB @ PhiP))

print(f"{'Fa':>5} {'Fb':>5} {'公式':>10} {'数値':>10}  一致")
for Fa, Fb in [(1.0,1.0),(0.25,0.25),(0.95,0.95),(0.9,0.8)]:
    f1, f2 = swap_formula(Fa,Fb), swap_numeric_phip(Fa,Fb)
    print(f"{Fa:5.2f} {Fb:5.2f} {f1:10.4f} {f2:10.4f}  {np.isclose(f1,f2)}")


端点に注目: `(1,1)→1`、`(1/4,1/4)→1/4`。両方完全なら結果も完全、両方最悪なら最悪のまま。
中間では **公式より単純な積 `Fa·Fb` を少しだけ上回る** (第2項のまぐれ当たり)。

> **W 表示の便利さ**: `W=(4F-1)/3` と置くと交換は単純な積 `W_swap = W_a·W_b` になります。
> 次の 演習4 で使います。

## 演習4: 鎖に連ねると 1/4 へ落ちる (→ スライド「連鎖で 1/4 へ」)

同じ忠実度 `F0` の区間を `n` 個、逐次交換で繋ぎます。`W` 表示の閉形

$$F_n = \frac{1 + 3 W^n}{4}, \qquad W = \frac{4F_0 - 1}{3}$$

を使うと一目瞭然です。

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
jp = fm.FontProperties(fname="/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc")

def F_to_W(F): return (4*F-1)/3
def W_to_F(W): return (1+3*W)/4
def F_chain(F0, n): return W_to_F(F_to_W(F0)**n)

ns = np.arange(1,33)
plt.figure(figsize=(7,4))
for F0 in [0.95, 0.9, 0.8]:
    plt.plot(ns, [F_chain(F0,n) for n in ns], "o-", ms=3, label=f"初期 F={F0}")
plt.axhline(0.25, color="crimson", ls="--", label="1/4 (完全混合)")
plt.xlabel("繋いだ区間数 n", fontproperties=jp)
plt.ylabel("端点間の忠実度 F", fontproperties=jp)
plt.legend(prop=jp); plt.grid(alpha=0.3); plt.ylim(0.2,1.0)
plt.title("交換を連ねると F は 1/4 へ漸近", fontproperties=jp)
plt.tight_layout(); plt.show()

print("初期F=0.9 の n=1,2,4,8,16,32:",
      [round(F_chain(0.9,n),4) for n in [1,2,4,8,16,32]])


距離は買えても品質を失う。**ただ繋ぐだけでは長距離量子通信は成立しない** ことが数値で出ました。
次に、落ちた品質を回復する道具 (純粋化) に進みます。その前に「F の落とし穴」を見ます。

## 演習5: 同じ F でも状態は一つではない (→ スライド「F の落とし穴」)

忠実度 `F` は状態を1つの数に潰した指標です。**同じ F でも中身は別物** であることを、
`F=0.8` の3状態 (X型・Z型・均等型) を構成して確かめます。

- **X型**: ビット反転 (`|Ψ⁺⟩` 方向) に残差が偏る
- **Z型**: 位相反転 (`|Φ⁻⟩` 方向) に残差が偏る
- **均等型** (脱分極/Werner): 残差が3方向に均等

In [ ]:
def bell_diag(a,b,c,d):
    "a|Φ⁺> + b|Φ⁻> + c|Ψ⁺> + d|Ψ⁻> の混合 (Bell対角状態)"
    return a*P_PhiP + b*P_PhiM + c*P_PsiP + d*P_PsiM

rho_X = bell_diag(0.8, 0.0, 0.2, 0.0)        # 残差は Ψ⁺ (ビット反転)
rho_Z = bell_diag(0.8, 0.2, 0.0, 0.0)        # 残差は Φ⁻ (位相反転)
rho_D = bell_diag(0.8, 0.2/3, 0.2/3, 0.2/3)  # 残差は均等 (脱分極)

for name, r in [("X型", rho_X), ("Z型", rho_Z), ("均等型", rho_D)]:
    print(f"{name}: F={fidelity(r):.4f}")

print("\nX型 と Z型 は同じ状態か?  ->", np.allclose(rho_X, rho_Z))
print("X型 と 均等型 は同じ状態か? ->", np.allclose(rho_X, rho_D))
print("\nX型の対角:", np.real(np.diag(rho_X)))
print("Z型の対角:", np.real(np.diag(rho_Z)))


3つとも `F=0.8` で一致しますが、密度行列は別物です。
次の純粋化は **誤りの特定の型を狙い撃つ** ので、同じ F でも型によって効き方が変わります。

> **壊して直す**: `rho_X` に対する別の忠実度 (例: `<Ψ⁺|ρ|Ψ⁺>`) を計算すると、
> X型では大きな値が出ます。F だけ見ていると見えない情報です。

## 演習6: 純粋化の写像 (→ スライド「純粋化の機構」「写像曲線と閾値」「天井 F*」)

純粋化は **雑音ペア2組を消費して、より高品質な1組を得る** 操作です (量→質)。
繰り返したときの忠実度は、入力 `F` から出力 `F'` への1本の写像で決まります。

ここでは標準的な **BBPSSW (Werner 入力)** の写像を実装し、

1. `F > 1/2` で 1 へ登り、`F < 1/2` で落ちる (**閾値 1/2**)
2. 純粋化操作自体に雑音があると、**天井 F\* < 1** ができる

を動かします。

> 注: 写像の係数は protocol (BBPSSW / DEJMPS) や twirl・CNOT 向きの規約で変わりますが、
> **定性結論 (閾値 1/2 と天井) は protocol に依りません**。DEJMPS は収束が速い変種です。

### 演習6-a: 写像の「形」と不動点を先に見る (→ スライド「写像曲線と閾値」「天井 F*」)

反復の前に、写像 `F -> F'` そのものの形を描きます。**`y = x` の線と交わる点が不動点** で、
そこでは純粋化しても F が変わりません。交点は `1/4`・`1/2`・`1` の3つ:

- `F = 1/4` … 完全混合。**安定** (落ちた先の床)
- `F = 1/2` … **不安定な閾値**。ここを境に上は 1 へ、下は 1/4 へ分かれる
- `F = 1`   … 理想ペア。**安定** (登った先の天井)

階段 (cobweb) を描くと、初期値によって登る/落ちるが分かれる様子が見えます。

In [ ]:
def bbpssw(F):
    "BBPSSW (Werner) 純粋化写像: 入力F -> 出力F'"
    a, b = F, (1-F)/3
    return (a*a + b*b) / (a*a + 2*a*b + 5*b*b)

Fg = np.linspace(0.25, 1.0, 400)
plt.figure(figsize=(6.0,6.0))
plt.plot(Fg, [bbpssw(f) for f in Fg], color="#0563C1", lw=2, label="純粋化写像 F'")
plt.plot([0.25,1.0],[0.25,1.0], color="gray", ls="--", label="y = x (不変の線)")
for fp in [0.25, 0.5, 1.0]:
    plt.plot(fp, fp, "o", color="crimson", ms=8, zorder=5)

def cobweb(F0, n, color, label):
    x = F0; px=[x]; py=[x]
    for _ in range(n):
        y = bbpssw(x); px += [x, y]; py += [y, y]; x = y
    plt.plot(px, py, color=color, lw=1.1, alpha=0.85, label=label)

cobweb(0.60, 8, "#2E7D32", "F0=0.60 -> 1 へ登る")
cobweb(0.42, 6, "#ED7D31", "F0=0.42 -> 1/4 へ落ちる")
plt.annotate("1/2", (0.5,0.5), textcoords="offset points", xytext=(9,-15),
             color="crimson", fontproperties=jp, fontsize=10)
plt.xlabel("入力 F", fontproperties=jp)
plt.ylabel("出力 F' = purify(F)", fontproperties=jp)
plt.title("純粋化写像と不動点: 1/2 が境目 (不安定)", fontproperties=jp)
plt.legend(prop=jp, loc="upper left", fontsize=9)
plt.grid(alpha=0.3); plt.xlim(0.25,1.0); plt.ylim(0.25,1.0)
plt.gca().set_aspect("equal"); plt.tight_layout(); plt.show()

print("不動点 (F'==F):", [f"{f:g}->{bbpssw(f):.4f}" for f in [0.25,0.5,1.0]])
print("1/2 のすぐ上下:",
      f"0.52->{bbpssw(0.52):.4f}(登る),", f"0.48->{bbpssw(0.48):.4f}(落ちる)")

In [ ]:
def bbpssw(F):
    "BBPSSW (Werner) の純粋化写像: 入力F -> 出力F'"
    a, b = F, (1-F)/3
    return (a*a + b*b) / (a*a + 2*a*b + 5*b*b)

# (1) 閾値 1/2 の確認
print("写像の動き:")
for F in [0.4, 0.5, 0.6, 0.8, 0.99, 1.0]:
    Fp = bbpssw(F)
    if abs(Fp-F) < 1e-9:
        tag = "不変 (不動点)"
    elif Fp > F:
        tag = "登る"
    else:
        tag = "落ちる"
    print(f"  F={F:.2f} -> F'={Fp:.4f}  [{tag}]")

# 反復してみる
def iterate(F, n, gate_noise=0.0):
    traj = [F]
    for _ in range(n):
        F = (1-gate_noise)*bbpssw(F) + gate_noise*0.25  # 雑音操作モデル
        traj.append(F)
    return traj

plt.figure(figsize=(7,4))
plt.plot(iterate(0.7, 10), "o-", label="F0=0.70 (完全操作)")
plt.plot(iterate(0.45,10), "s-", label="F0=0.45 (閾値下)")
plt.plot(iterate(0.7, 10, gate_noise=0.05), "^-", label="F0=0.70 (雑音操作 g=0.05)")
plt.axhline(0.5, color="crimson", ls="--", label="閾値 1/2")
plt.axhline(1.0, color="green", ls=":", alpha=0.6)
plt.xlabel("純粋化の反復回数", fontproperties=jp)
plt.ylabel("忠実度 F", fontproperties=jp)
plt.legend(prop=jp); plt.grid(alpha=0.3)
plt.title("純粋化: F>1/2 は登り、雑音操作だと天井で頭打ち", fontproperties=jp)
plt.tight_layout(); plt.show()


- `F0=0.70` は反復ごとに 1 へ近づく
- `F0=0.45` (1/2 未満) は逆に落ちていく → **スタートが 1/2 を超えていないと純粋化は機能しない**
- 雑音操作 (`g=0.05`) は途中で頭打ち → **完全な 1 には決して届かない (天井 F\*)**

これが先週の **誤り訂正 (操作エラー率の閾値・任意抑制)** と決定的に違う点です。
純粋化は **状態忠実度の閾値 + 天井**。同じ「雑音と戦う」でも軸も保証も別物で、
「ベルペア版の誤り訂正」と同一視してはいけません。

> **やってみる**: `gate_noise` を 0.02 / 0.10 に変えて天井がどこに来るか見てください。
> `g=0.10` 付近では天井が大きく下がり、純粋化の旨味が消えます。

## 演習7: 入れ子 — 交換と純粋化を交互に (→ スライド「相反する二手と入れ子」「ノコギリ波と代金」)

相反する二つの手を **入れ子** にして回します。

- **交換**: 距離を倍にするが忠実度を下げる
- **純粋化**: 忠実度を上げるがペアを消費する

各レベルで「交換して距離を倍 → 純粋化で 1/2 の上へ戻す」を繰り返すと、
忠実度は **ノコギリ波** を描きながら 1/2 線の上に留まり、距離だけが伸びていきます。
同時に **消費ペア数 (代金)** が増えていくのが見えます。

In [ ]:
def swap_formula(Fa, Fb):
    return Fa*Fb + (1-Fa)*(1-Fb)/3

F = 0.95
target = 0.90       # 各レベルで純粋化で戻す目標
levels = 6

xs, Fs = [0.0], [F]
cost_x, cost_y = [], []
x, total_pairs = 0.0, 1

for L in range(1, levels+1):
    # 交換: 同一 F の2本を繋ぐ (距離が倍に)
    x += 0.5; F = swap_formula(F, F); total_pairs *= 2
    xs.append(x); Fs.append(F)
    # 純粋化: target まで戻す
    while F < target:
        x += 0.25; F = bbpssw(F); total_pairs *= 2
        xs.append(x); Fs.append(F)
    cost_x.append(x); cost_y.append(total_pairs)

fig, ax1 = plt.subplots(figsize=(7.5,4.2))
ax1.plot(xs, Fs, "o-", color="#0563C1", ms=3, label="忠実度 F")
ax1.axhline(0.5, color="crimson", ls="--")
ax1.set_xlabel("入れ子レベルの進行 (swap↓ / purify↑)", fontproperties=jp)
ax1.set_ylabel("忠実度 F", fontproperties=jp, color="#0563C1")
ax1.set_ylim(0.45,1.0)
ax2 = ax1.twinx()
ax2.plot(cost_x, cost_y, "s--", color="#ED7D31", label="累積消費ペア数")
ax2.set_ylabel("累積 消費ペア数 (代金)", fontproperties=jp, color="#ED7D31")
ax2.set_yscale("log")
ax1.set_title("入れ子: 1/2 線の上に留まりつつ距離を伸ばす (代金つき)", fontproperties=jp)
plt.tight_layout(); plt.show()

print(f"到達した区間数 = 2^{levels} = {2**levels}")
print("F は一度も 1/2 を割らずに距離を伸ばせた:", all(f > 0.5 for f in Fs))


**読み取り**: 忠実度 (青) は交換で下がり純粋化で戻る、を繰り返して **1/2 線の上に留まる**。
一方で消費ペア数 (オレンジ) は右肩上がり。**「動くこと」と「その代金」を同時に体感** できます。

レベルを1つ上げるごとに到達距離が倍 → レベル `L` で `2^L` 区間。
深さ (レベル数) は距離の **対数**、コストは距離の **多項式** で増える、というのが
Briegel らの Purify and Swap の骨子です (コストの正確な次数は副読本)。

> **これが今日の道具立ての全体像**: もつれ生成 + 交換 + 純粋化 を入れ子で回す。
> 残る論点 (古典通信の往復・量子メモリ・中継器の世代) はスライドで扱います。